In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split,GridSearchCV
import optuna 
from sklearn.metrics import r2_score,mean_squared_error,accuracy_score,precision_score,classification_report

In [2]:
data=pd.read_csv("../Dataset/After_preprocessing.csv")

USAGE OF MUTUAL INFORMATION FOR FEATURE SELECTION FOR SQUAT 3

In [3]:
# from sklearn.model_selection import train_test_split
# from sklearn.feature_selection import SelectKBest,mutual_info_regression
# from sklearn.ensemble import GradientBoostingRegressor
# from sklearn.metrics import r2_score

# x=data.drop(columns=['Squat3','Actual_Squat3','Failed_Squat3'])
# y=data['Squat3']
# x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

In [8]:
# To directly check the MI of each feature
# MI=mutual_info_regression(x_train,y_train)
# MI_values=pd.Series(MI)
# MI_values.index=x_train.columns
# MI_values.sort_values(ascending=False)

Actual_Squat2         2.859425
Squat2                2.769024
Actual_Squat1         2.311051
Squat1                2.278714
Total                 1.758526
Deadlift2             1.180357
Actual_Deadlift2      1.170530
Actual_Deadlift3      1.135833
Deadlift3             1.109936
Deadlift1             1.081991
Actual_Deadlift1      1.069391
Benchpress2           1.068057
Benchpress3           1.056345
Actual_Benchpress2    1.048545
Actual_Benchpress1    1.045503
Benchpress1           1.037175
Actual_Benchpress3    1.017175
Weight                0.464393
Gender                0.401984
Age                   0.183652
Failed_Deadlift3      0.020610
Failed_Benchpress2    0.002995
Failed_Benchpress3    0.001364
Failed_Deadlift1      0.000000
Failed_Benchpress1    0.000000
Failed_Deadlift2      0.000000
Failed_Squat2         0.000000
Failed_Squat1         0.000000
dtype: float64

In [12]:
# To get the top 5 features with highest MI
# selector=SelectKBest(mutual_info_regression,k=5)
# selector.fit(x_train,y_train)
# x_train.columns[selector.get_support()]
# x_train=selector.transform(x_train)
# x_test=selector.transform(x_test)

Index(['Squat1', 'Squat2', 'Total', 'Actual_Squat1', 'Actual_Squat2'], dtype='object')

In [18]:
#To get the top k features when we dont know what the best value of k could be 
# scores=[]
# for i in range(1,len(x.columns)):
#     selector=SelectKBest(mutual_info_regression,k=i)
#     selector.fit(x_train,y_train)

#     select_x_train=selector.transform(x_train)
#     select_x_test=selector.transform(x_test)
#     gb=GradientBoostingRegressor()
#     gb.fit(select_x_train,y_train)
#     pred=gb.predict(select_x_test)
#     scores.append(r2_score(y_test,pred))

# print(scores)

[0.8441088204049033, 0.8439630018208832, 0.8400545760267153, 0.839915436719325, 0.8266620552158653, 0.8496194531602793, 0.848027277484627, 0.8464899757884883, 0.8382724486156357, 0.8374392692732924, 0.840385140543738, 0.8366622550981032, 0.841112444582123, 0.8352160195076715, 0.843647421613646, 0.8476099837748501, 0.8435364451077407, 0.8407975336407975, 0.8400435464487941, 0.8421509663790494, 0.8402917543099544, 0.8421375716362898, 0.8386570805927713, 0.839999734318072, 0.841286738875279, 0.8419623563101024, 0.8429647475328622]


In [3]:
correlation=data.corr(method='spearman')

## Predicting the 2nd lift (Squat)

In [4]:
correlation['Squat2'].sort_values(ascending=False)

Squat2                1.000000
Squat1                0.997670
Squat3                0.986284
Total                 0.958687
Deadlift2             0.942813
Deadlift1             0.942623
Deadlift3             0.941176
Benchpress2           0.931557
Benchpress1           0.930307
Benchpress3           0.904085
Actual_Deadlift1      0.876539
Actual_Benchpress1    0.822677
Actual_Squat1         0.819930
Actual_Deadlift2      0.787728
Gender                0.762701
Actual_Benchpress2    0.725738
Weight                0.696885
Actual_Squat2         0.646109
Actual_Benchpress3    0.237803
Actual_Squat3         0.229240
Failed_Deadlift3      0.148143
Failed_Squat3         0.061541
Actual_Deadlift3      0.042342
Failed_Benchpress1    0.033894
Failed_Deadlift2     -0.005185
Failed_Squat2        -0.032364
Failed_Squat1        -0.055822
Failed_Benchpress2   -0.065838
Failed_Deadlift1     -0.073131
Failed_Benchpress3   -0.085485
Age                  -0.090091
Name: Squat2, dtype: float64

In [5]:
x=data[['Gender','Weight','Squat1']]
y=data['Squat2']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

In [ ]:
def objective(trail,data=x,target=y):
    param={
       'loss':trail.suggest_categorical('loss',['squared_error', 'absolute_error', 'huber']),
        "learning_rate":trail.suggest_categorical("learning_rate",[.00001,.0003,.008,.02,.01,1]),
        "n_estimators":trail.suggest_categorical('n_estimators',[50,100,300,500]),
        "max_depth":trail.suggest_categorical('max_depth',[8,16,32,64,128]),
        "min_samples_leaf":trail.suggest_categorical('min_samples_leaf',[8,16,32,64,128])
    }
    GB_squat2=GradientBoostingRegressor(**param)
    GB_squat2.fit(x_train,y_train)
    y_pred=GB_squat2.predict(x_test)
    return r2_score(y_test,y_pred)

trials=optuna.create_study(direction='maximize')
trials.optimize(objective,n_trials=5)

[I 2024-10-29 00:36:53,332] A new study created in memory with name: no-name-5cbb7164-9a4b-4e04-a397-d40642f6bf8f
[I 2024-10-29 00:36:59,956] Trial 0 finished with value: 0.9895441485026376 and parameters: {'loss': 'absolute_error', 'learning_rate': 0.02, 'n_estimators': 500, 'max_depth': 32, 'min_samples_leaf': 16}. Best is trial 0 with value: 0.9895441485026376.
[I 2024-10-29 00:37:00,418] Trial 1 finished with value: 0.5431449141448835 and parameters: {'loss': 'huber', 'learning_rate': 0.008, 'n_estimators': 50, 'max_depth': 16, 'min_samples_leaf': 32}. Best is trial 0 with value: 0.9895441485026376.
[I 2024-10-29 00:37:03,585] Trial 2 finished with value: 0.9841945515193883 and parameters: {'loss': 'absolute_error', 'learning_rate': 1, 'n_estimators': 500, 'max_depth': 32, 'min_samples_leaf': 32}. Best is trial 0 with value: 0.9895441485026376.
[I 2024-10-29 00:37:03,833] Trial 3 finished with value: 0.45196234396692603 and parameters: {'loss': 'absolute_error', 'learning_rate': 0.

In [10]:
params=trials.best_trial.params
GB_squat2=GradientBoostingRegressor(**params)
GB_squat2.fit(x_train,y_train)
print(f"Training score for Gradientboosting regressor: {GB_squat2.score(x_train,y_train)}")
y_pred=GB_squat2.predict(x_test)
print(f"Testing score for Gradientboosting regressor: {r2_score(y_test,y_pred)}")
print(f"Mean squred error:{mean_squared_error(y_test,y_pred)}")

Training score for Gradientboosting regressor: 0.9932674626189669
Testing score for Gradientboosting regressor: 0.9919569990945406
Mean squred error:29.821982019802565


In [15]:
pickle.dump(GB_squat2,open('../Models/squat2_predictor.sav','wb'))

## Predicting the 2nd lift (Bench)

In [11]:
correlation['Benchpress2'].sort_values(ascending=False)

Benchpress2           1.000000
Benchpress1           0.998577
Benchpress3           0.962858
Total                 0.939924
Squat2                0.931557
Squat1                0.928757
Squat3                0.923015
Deadlift2             0.919789
Deadlift3             0.918790
Deadlift1             0.911898
Actual_Benchpress1    0.885792
Actual_Deadlift1      0.854590
Gender                0.811943
Actual_Benchpress2    0.770405
Actual_Squat1         0.760760
Actual_Deadlift2      0.759499
Weight                0.657614
Actual_Squat2         0.615959
Actual_Squat3         0.245724
Actual_Benchpress3    0.216361
Failed_Deadlift3      0.137598
Actual_Deadlift3      0.041009
Failed_Benchpress1    0.035294
Failed_Squat3         0.010929
Failed_Deadlift2      0.004969
Failed_Benchpress2   -0.050154
Failed_Squat1        -0.052483
Failed_Benchpress3   -0.053205
Failed_Squat2        -0.057700
Age                  -0.062969
Failed_Deadlift1     -0.095717
Name: Benchpress2, dtype: float64

In [12]:
x=data[['Gender','Weight','Benchpress1']]
y=data['Benchpress2']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

In [ ]:
def objective(trail,data=x,target=y):
    param={
       'loss':trail.suggest_categorical('loss',['squared_error', 'absolute_error', 'huber']),
        "learning_rate":trail.suggest_categorical("learning_rate",[.00001,.0003,.008,.02,.01,1]),
        "n_estimators":trail.suggest_categorical('n_estimators',[50,100,300,500]),
        "max_depth":trail.suggest_categorical('max_depth',[8,16,32,64,128]),
        "min_samples_leaf":trail.suggest_categorical('min_samples_leaf',[8,16,32,64,128])
    }
    GB_bench2=GradientBoostingRegressor(**param)
    GB_bench2.fit(x_train,y_train)
    y_pred=GB_bench2.predict(x_test)
    return r2_score(y_test,y_pred)

trials=optuna.create_study(direction='maximize')
trials.optimize(objective,n_trials=5)

[I 2024-10-29 00:37:31,235] A new study created in memory with name: no-name-b1503803-aca4-4fa9-ad10-70e048abb4ff
[I 2024-10-29 00:37:36,284] Trial 0 finished with value: 0.9930488912792104 and parameters: {'loss': 'absolute_error', 'learning_rate': 0.01, 'n_estimators': 500, 'max_depth': 8, 'min_samples_leaf': 16}. Best is trial 0 with value: 0.9930488912792104.
[I 2024-10-29 00:37:36,466] Trial 1 finished with value: 0.9650596666612229 and parameters: {'loss': 'huber', 'learning_rate': 1, 'n_estimators': 50, 'max_depth': 64, 'min_samples_leaf': 128}. Best is trial 0 with value: 0.9930488912792104.
[I 2024-10-29 00:37:36,612] Trial 2 finished with value: -0.04225239569859429 and parameters: {'loss': 'absolute_error', 'learning_rate': 0.0003, 'n_estimators': 50, 'max_depth': 128, 'min_samples_leaf': 64}. Best is trial 0 with value: 0.9930488912792104.
[I 2024-10-29 00:37:40,808] Trial 3 finished with value: 0.9888308864971418 and parameters: {'loss': 'huber', 'learning_rate': 1, 'n_est

In [14]:
params=trials.best_trial.params
GB_bench2=GradientBoostingRegressor(**params)
GB_bench2.fit(x_train,y_train)
print(f"Training score for Gradientboosting regressor: {GB_bench2.score(x_train,y_train)}")
y_pred=GB_bench2.predict(x_test)
print(f"Testing score for Gradientboosting regressor: {r2_score(y_test,y_pred)}")
print(f"Mean squred error:{mean_squared_error(y_test,y_pred)}")

Training score for Gradientboosting regressor: 0.9946671420725237
Testing score for Gradientboosting regressor: 0.9930642517256436
Mean squred error:12.389069053524889


In [16]:
pickle.dump(GB_bench2,open('../Models/bench2_predictor.sav','wb'))

## Predicting the 2nd lift (Deadlift)

In [17]:
correlation['Deadlift2'].sort_values(ascending=False)

Deadlift2             1.000000
Deadlift3             0.997101
Deadlift1             0.994214
Total                 0.951226
Squat2                0.942813
Squat1                0.941421
Squat3                0.933070
Actual_Deadlift1      0.926491
Benchpress2           0.919789
Benchpress1           0.918361
Benchpress3           0.888759
Actual_Deadlift2      0.820863
Actual_Benchpress1    0.810973
Gender                0.805298
Actual_Squat1         0.760915
Actual_Benchpress2    0.718117
Weight                0.638358
Actual_Squat2         0.624861
Actual_Squat3         0.248922
Actual_Benchpress3    0.246908
Failed_Deadlift3      0.186572
Failed_Benchpress1    0.040832
Failed_Squat3         0.031883
Failed_Deadlift2      0.020342
Actual_Deadlift3      0.017247
Failed_Squat1        -0.030233
Failed_Squat2        -0.051925
Failed_Benchpress2   -0.064294
Age                  -0.071331
Failed_Deadlift1     -0.082358
Failed_Benchpress3   -0.095828
Name: Deadlift2, dtype: float64

In [28]:
x=data[['Gender','Weight','Deadlift1']]
y=data['Deadlift2']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

In [ ]:
def objective(trail,data=x,target=y):
    param={
       'loss':trail.suggest_categorical('loss',['squared_error', 'absolute_error', 'huber']),
        "learning_rate":trail.suggest_categorical("learning_rate",[.00001,.0003,.008,.02,.01,1]),
        "n_estimators":trail.suggest_categorical('n_estimators',[50,100,300,500]),
        "max_depth":trail.suggest_categorical('max_depth',[8,16,32,64,128]),
        "min_samples_leaf":trail.suggest_categorical('min_samples_leaf',[8,16,32,64,128])
    }
    GB_Deadlift2=GradientBoostingRegressor(**param)
    GB_Deadlift2.fit(x_train,y_train)
    y_pred=GB_Deadlift2.predict(x_test)
    return r2_score(y_test,y_pred)

trials=optuna.create_study(direction='maximize')
trials.optimize(objective,n_trials=5)

[I 2024-10-29 00:41:47,067] A new study created in memory with name: no-name-21a57279-b043-4f39-a8eb-ab046dd56e7c
[I 2024-10-29 00:41:49,370] Trial 0 finished with value: 0.9715811148570415 and parameters: {'loss': 'absolute_error', 'learning_rate': 0.01, 'n_estimators': 500, 'max_depth': 16, 'min_samples_leaf': 64}. Best is trial 0 with value: 0.9715811148570415.
[I 2024-10-29 00:41:54,137] Trial 1 finished with value: 0.9848385082537497 and parameters: {'loss': 'huber', 'learning_rate': 0.01, 'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 16}. Best is trial 1 with value: 0.9848385082537497.
[I 2024-10-29 00:41:55,737] Trial 2 finished with value: -0.09078339314001127 and parameters: {'loss': 'absolute_error', 'learning_rate': 1e-05, 'n_estimators': 300, 'max_depth': 64, 'min_samples_leaf': 8}. Best is trial 1 with value: 0.9848385082537497.
[I 2024-10-29 00:41:55,870] Trial 3 finished with value: 0.04546263278342577 and parameters: {'loss': 'squared_error', 'learning_rate':

In [30]:
params=trials.best_trial.params
GB_Deadlift2=GradientBoostingRegressor(**params)
GB_Deadlift2.fit(x_train,y_train)
print(f"Training score for Gradientboosting regressor: {GB_Deadlift2.score(x_train,y_train)}")
y_pred=GB_Deadlift2.predict(x_test)
print(f"Testing score for Gradientboosting regressor: {r2_score(y_test,y_pred)}")
print(f"Mean squred error:{mean_squared_error(y_test,y_pred)}")

Training score for Gradientboosting regressor: 0.9862291262335837
Testing score for Gradientboosting regressor: 0.9848677005702421
Mean squred error:58.25538177476889


In [31]:
pickle.dump(GB_bench2,open('../Models/Deadlift2_predictor.sav','wb'))

## Predicting the 3rd lift (Squat)

In [32]:
correlation['Squat3'].sort_values(ascending=False)

Squat3                1.000000
Squat2                0.986284
Squat1                0.982920
Total                 0.951353
Deadlift2             0.933070
Deadlift3             0.932751
Deadlift1             0.931491
Benchpress2           0.923015
Benchpress1           0.921163
Benchpress3           0.896186
Actual_Deadlift1      0.866074
Actual_Benchpress1    0.814868
Actual_Squat1         0.803713
Actual_Deadlift2      0.778248
Gender                0.755005
Actual_Benchpress2    0.731444
Weight                0.692182
Actual_Squat2         0.655537
Actual_Squat3         0.238692
Actual_Benchpress3    0.235620
Failed_Deadlift3      0.142115
Failed_Squat3         0.064101
Actual_Deadlift3      0.045363
Failed_Benchpress1    0.031740
Failed_Deadlift2     -0.002421
Failed_Squat1        -0.050969
Failed_Squat2        -0.061764
Failed_Deadlift1     -0.070869
Failed_Benchpress2   -0.083921
Failed_Benchpress3   -0.084812
Age                  -0.092502
Name: Squat3, dtype: float64

In [33]:
x=data[['Gender','Weight','Squat1','Squat2']]
y=data['Squat3']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

In [ ]:
def objective(trail,data=x,target=y):
    param={
       'loss':trail.suggest_categorical('loss',['squared_error', 'absolute_error', 'huber']),
        "learning_rate":trail.suggest_categorical("learning_rate",[.00001,.0003,.008,.02,.01,1]),
        "n_estimators":trail.suggest_categorical('n_estimators',[50,100,300,500]),
        "max_depth":trail.suggest_categorical('max_depth',[8,16,32,64,128]),
    }
    GB_squat3=GradientBoostingRegressor(**param)
    GB_squat3.fit(x_train,y_train)
    y_pred=GB_squat3.predict(x_test)
    return r2_score(y_test,y_pred)

trials=optuna.create_study(direction='maximize')
trials.optimize(objective,n_trials=5)

[I 2024-10-29 00:42:17,739] A new study created in memory with name: no-name-e214e5cb-dce3-44e7-bbe0-e1a2650e169f
[I 2024-10-29 00:42:43,172] Trial 0 finished with value: 0.9795582003199008 and parameters: {'loss': 'absolute_error', 'learning_rate': 0.008, 'n_estimators': 500, 'max_depth': 64}. Best is trial 0 with value: 0.9795582003199008.
[I 2024-10-29 00:43:06,787] Trial 1 finished with value: 0.9789840940290531 and parameters: {'loss': 'absolute_error', 'learning_rate': 0.008, 'n_estimators': 500, 'max_depth': 64}. Best is trial 0 with value: 0.9795582003199008.
[I 2024-10-29 00:43:21,526] Trial 2 finished with value: 0.042562147554466634 and parameters: {'loss': 'huber', 'learning_rate': 0.0003, 'n_estimators': 100, 'max_depth': 16}. Best is trial 0 with value: 0.9795582003199008.
[I 2024-10-29 00:43:32,914] Trial 3 finished with value: 0.9680138051361831 and parameters: {'loss': 'absolute_error', 'learning_rate': 0.01, 'n_estimators': 300, 'max_depth': 128}. Best is trial 0 with

In [35]:
params=trials.best_trial.params
GB_squat3=GradientBoostingRegressor(**params)
GB_squat3.fit(x_train,y_train)
print(f"Training score for Gradientboosting regressor: {GB_squat3.score(x_train,y_train)}")
y_pred=GB_squat3.predict(x_test)
print(f"Testing score for Gradientboosting regressor: {r2_score(y_test,y_pred)}")
print(f"Mean squred error:{mean_squared_error(y_test,y_pred)}")

Training score for Gradientboosting regressor: 0.9994448393484883
Testing score for Gradientboosting regressor: 0.9923656284913315
Mean squred error:31.858149360940175


In [37]:
pickle.dump(GB_squat3,open('../Models/squat3_predictor.sav','wb'))

## Predicting the 3rd lift (Bench)

In [41]:
correlation['Benchpress3'].sort_values(ascending=False)

Benchpress3           1.000000
Benchpress2           0.962858
Benchpress1           0.960384
Total                 0.909957
Squat2                0.904085
Squat1                0.900890
Squat3                0.896186
Deadlift2             0.888759
Deadlift3             0.888308
Deadlift1             0.880855
Actual_Benchpress1    0.847783
Actual_Deadlift1      0.823324
Gender                0.782396
Actual_Benchpress2    0.772340
Actual_Squat1         0.740504
Actual_Deadlift2      0.736069
Weight                0.639667
Actual_Squat2         0.603385
Actual_Squat3         0.254109
Actual_Benchpress3    0.222213
Failed_Deadlift3      0.128306
Actual_Deadlift3      0.041587
Failed_Benchpress1    0.038744
Failed_Deadlift2      0.000014
Failed_Squat3        -0.003878
Failed_Benchpress3   -0.023787
Age                  -0.045967
Failed_Squat1        -0.056374
Failed_Squat2        -0.059407
Failed_Deadlift1     -0.085750
Failed_Benchpress2   -0.096871
Name: Benchpress3, dtype: float64

In [51]:
x=data[['Gender','Weight','Benchpress1','Benchpress2']]
y=data['Benchpress3']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

In [ ]:
def objective(trail,data=x,target=y):
    param={
       'loss':trail.suggest_categorical('loss',['squared_error', 'absolute_error', 'huber']),
        "learning_rate":trail.suggest_categorical("learning_rate",[.00001,.0003,.008,.02,.01,1]),
        "n_estimators":trail.suggest_categorical('n_estimators',[50,100,300,500]),
        "max_depth":trail.suggest_categorical('max_depth',[8,16,32,64,128]),
        "min_samples_leaf":trail.suggest_categorical('min_samples_leaf',[8,16,32,64,128])
    }
    GB_bench3=GradientBoostingRegressor(**param)
    GB_bench3.fit(x_train,y_train)
    y_pred=GB_bench3.predict(x_test)
    return r2_score(y_test,y_pred)

trials=optuna.create_study(direction='maximize')
trials.optimize(objective,n_trials=5)

[I 2024-10-29 00:48:01,448] A new study created in memory with name: no-name-edf8185c-c4a8-400b-9e1b-a08d6bd5bc91
[I 2024-10-29 00:48:05,370] Trial 0 finished with value: 0.9223549911869036 and parameters: {'loss': 'absolute_error', 'learning_rate': 0.01, 'n_estimators': 500, 'max_depth': 32, 'min_samples_leaf': 32}. Best is trial 0 with value: 0.9223549911869036.
[I 2024-10-29 00:48:05,651] Trial 1 finished with value: 0.9020101090859348 and parameters: {'loss': 'squared_error', 'learning_rate': 0.02, 'n_estimators': 100, 'max_depth': 128, 'min_samples_leaf': 16}. Best is trial 0 with value: 0.9223549911869036.
[I 2024-10-29 00:48:05,837] Trial 2 finished with value: 0.0011169650389338814 and parameters: {'loss': 'squared_error', 'learning_rate': 1e-05, 'n_estimators': 100, 'max_depth': 8, 'min_samples_leaf': 64}. Best is trial 0 with value: 0.9223549911869036.
[I 2024-10-29 00:48:08,672] Trial 3 finished with value: 0.1389720892993076 and parameters: {'loss': 'huber', 'learning_rate'

In [53]:
params=trials.best_trial.params
GB_bench3=GradientBoostingRegressor(**params)
GB_bench3.fit(x_train,y_train)
print(f"Training score for Gradientboosting regressor: {GB_bench3.score(x_train,y_train)}")
y_pred=GB_bench3.predict(x_test)
print(f"Testing score for Gradientboosting regressor: {r2_score(y_test,y_pred)}")
print(f"Mean squred error:{mean_squared_error(y_test,y_pred)}")

Training score for Gradientboosting regressor: 0.8511269105779841
Testing score for Gradientboosting regressor: 0.9220734059967475
Mean squred error:174.1402670638271


In [54]:
pickle.dump(GB_squat3,open('../Models/bench3_predictor.sav','wb'))

# Predicting the 3rd lift (Deadlift)

In [55]:
correlation['Deadlift3'].sort_values(ascending=False)

Deadlift3             1.000000
Deadlift2             0.997101
Deadlift1             0.989739
Total                 0.950724
Squat2                0.941176
Squat1                0.938991
Squat3                0.932751
Actual_Deadlift1      0.920801
Benchpress2           0.918790
Benchpress1           0.917098
Benchpress3           0.888308
Actual_Deadlift2      0.833662
Actual_Benchpress1    0.809715
Gender                0.806872
Actual_Squat1         0.759999
Actual_Benchpress2    0.718193
Weight                0.639942
Actual_Squat2         0.626964
Actual_Benchpress3    0.252742
Actual_Squat3         0.250002
Failed_Deadlift3      0.184728
Failed_Benchpress1    0.041841
Failed_Squat3         0.029699
Actual_Deadlift3      0.019421
Failed_Deadlift2     -0.009818
Failed_Squat1        -0.031475
Failed_Squat2        -0.052974
Failed_Benchpress2   -0.067907
Failed_Deadlift1     -0.076204
Age                  -0.077542
Failed_Benchpress3   -0.102071
Name: Deadlift3, dtype: float64

In [66]:
x=data[['Gender','Weight','Deadlift1','Deadlift2']]
y=data['Deadlift3']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

In [ ]:
def objective(trail,data=x,target=y):
    param={
       'loss':trail.suggest_categorical('loss',['squared_error', 'absolute_error', 'huber']),
        "learning_rate":trail.suggest_categorical("learning_rate",[.00001,.0003,.008,.02,.01,1]),
        "n_estimators":trail.suggest_categorical('n_estimators',[50,100,300,500]),
        "max_depth":trail.suggest_categorical('max_depth',[8,16,32,64,128]),
    }
    GB_deadlift3=GradientBoostingRegressor(**param)
    GB_deadlift3.fit(x_train,y_train)
    y_pred=GB_deadlift3.predict(x_test)
    return r2_score(y_test,y_pred)

trials=optuna.create_study(direction='maximize')
trials.optimize(objective,n_trials=5)

[I 2024-10-29 00:53:46,930] A new study created in memory with name: no-name-773827b8-32f3-4818-a6f5-ee067691b4a8
[I 2024-10-29 00:53:48,042] Trial 0 finished with value: 0.9838315863917396 and parameters: {'loss': 'huber', 'learning_rate': 1, 'n_estimators': 500, 'max_depth': 128}. Best is trial 0 with value: 0.9838315863917396.
[I 2024-10-29 00:54:21,040] Trial 1 finished with value: 0.987020260183942 and parameters: {'loss': 'absolute_error', 'learning_rate': 0.02, 'n_estimators': 500, 'max_depth': 32}. Best is trial 1 with value: 0.987020260183942.
[I 2024-10-29 00:54:39,895] Trial 2 finished with value: 0.9868638692093837 and parameters: {'loss': 'absolute_error', 'learning_rate': 0.02, 'n_estimators': 500, 'max_depth': 16}. Best is trial 1 with value: 0.987020260183942.
[I 2024-10-29 00:55:55,253] Trial 3 finished with value: 0.98396799533217 and parameters: {'loss': 'huber', 'learning_rate': 0.01, 'n_estimators': 500, 'max_depth': 128}. Best is trial 1 with value: 0.987020260183

In [69]:
params=trials.best_trial.params
GB_deadlift3=GradientBoostingRegressor(**params)
GB_deadlift3.fit(x_train,y_train)
print(f"Training score for Gradientboosting regressor: {GB_deadlift3.score(x_train,y_train)}")
y_pred=GB_deadlift3.predict(x_test)
print(f"Testing score for Gradientboosting regressor: {r2_score(y_test,y_pred)}")
print(f"Mean squred error:{mean_squared_error(y_test,y_pred)}")

Training score for Gradientboosting regressor: 0.9999151885011386
Testing score for Gradientboosting regressor: 0.9874227257283912
Mean squred error:53.33299805694129


In [70]:
pickle.dump(GB_deadlift3,open("../Models/deadlift3_predictor.sav",'wb'))

## Predicting if third lift is a successful lift (Squat)

In [12]:
correlation['Failed_Squat3'].sort_values(ascending=False)[:15]

Failed_Squat3         1.000000
Failed_Deadlift3      0.089634
Failed_Benchpress2    0.088913
Failed_Squat1         0.081538
Failed_Squat2         0.078541
Failed_Deadlift1      0.073743
Squat1                0.071825
Squat3                0.069326
Squat2                0.066717
Failed_Benchpress1    0.063511
Failed_Deadlift2      0.056107
Failed_Benchpress3    0.052414
Deadlift1             0.043648
Deadlift2             0.035494
Deadlift3             0.033113
Name: Failed_Squat3, dtype: float64

In [53]:
x=data[['Failed_Squat1','Failed_Squat2','Failed_Deadlift3']]
y=data['Failed_Squat3']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

In [ ]:
def objective(trail,data=x,target=y):
    param={
        "criterion":trail.suggest_categorical("criterion",["friedman_mse", "squared_error"]),
        "learning_rate":trail.suggest_categorical("learning_rate",[.00001,.0003,.008,.02,.01,1]),
        "n_estimators":trail.suggest_categorical('n_estimators',[50,100,300,500]),
        "max_depth":trail.suggest_categorical('max_depth',[8,16,32,64,128]),
        "min_samples_leaf":trail.suggest_categorical('min_samples_leaf',[8,16,32,64,128])
    }
    GB_status_squat3=GradientBoostingClassifier(**param)
    GB_status_squat3.fit(x_train,y_train)
    y_pred=GB_status_squat3.predict(x_test)
    return accuracy_score(y_test,y_pred)

trials=optuna.create_study(direction='maximize')
trials.optimize(objective,n_trials=5)

[I 2024-10-26 21:37:17,072] A new study created in memory with name: no-name-0726674a-5820-409e-81eb-91f588b50d7e
[I 2024-10-26 21:37:17,151] Trial 0 finished with value: 0.6524064171122995 and parameters: {'criterion': 'squared_error', 'learning_rate': 0.01, 'n_estimators': 100, 'max_depth': 8, 'min_samples_leaf': 16}. Best is trial 0 with value: 0.6524064171122995.
[I 2024-10-26 21:37:17,321] Trial 1 finished with value: 0.679144385026738 and parameters: {'criterion': 'friedman_mse', 'learning_rate': 0.02, 'n_estimators': 300, 'max_depth': 128, 'min_samples_leaf': 128}. Best is trial 1 with value: 0.679144385026738.
[I 2024-10-26 21:37:17,361] Trial 2 finished with value: 0.6363636363636364 and parameters: {'criterion': 'friedman_mse', 'learning_rate': 1, 'n_estimators': 50, 'max_depth': 64, 'min_samples_leaf': 8}. Best is trial 1 with value: 0.679144385026738.
[I 2024-10-26 21:37:17,405] Trial 3 finished with value: 0.679144385026738 and parameters: {'criterion': 'friedman_mse', 'le

In [55]:
params=trials.best_trial.params
GB_status_squat3=GradientBoostingClassifier(**params)
GB_status_squat3.fit(x_train,y_train)
print(f"Training score for Gradientboosting classifier: {GB_status_squat3.score(x_train,y_train)}")
y_pred=GB_status_squat3.predict(x_test)
print(f"Testing score for Gradientboosting classifier: {accuracy_score(y_test,y_pred)}")

Training score for Gradientboosting classifier: 0.6335570469798658
Testing score for Gradientboosting classifier: 0.679144385026738


In [47]:
print(f"Classification report:{classification_report(y_test,y_pred,zero_division=1.0)}")

Classification report:              precision    recall  f1-score   support

           0       0.67      1.00      0.80       125
           1       1.00      0.00      0.00        62

    accuracy                           0.67       187
   macro avg       0.83      0.50      0.40       187
weighted avg       0.78      0.67      0.54       187



In [82]:
pickle.dump(GB_status_squat3,open('../Models/squat3_status_predictor.sav','wb'))

## Predicting if third lift is a successful lift (Bench)

In [8]:
correlation['Failed_Benchpress3'].sort_values(ascending=False)[:15]

Failed_Benchpress3    1.000000
Failed_Benchpress2    0.120484
Failed_Squat2         0.069264
Failed_Squat1         0.056485
Failed_Squat3         0.052414
Age                   0.039625
Failed_Deadlift3      0.032673
Failed_Deadlift2      0.029433
Failed_Deadlift1      0.007338
Failed_Benchpress1   -0.002631
Benchpress3          -0.005431
Actual_Deadlift3     -0.033186
Actual_Benchpress1   -0.033792
Gender               -0.035345
Benchpress2          -0.048580
Name: Failed_Benchpress3, dtype: float64

In [9]:
correlation['Failed_Benchpress3'].sort_values(ascending=False)[::-1][:15]

Actual_Benchpress3   -0.925639
Actual_Benchpress2   -0.127185
Deadlift3            -0.101966
Actual_Squat2        -0.099391
Deadlift2            -0.096415
Total                -0.095286
Deadlift1            -0.088300
Actual_Squat1        -0.083410
Squat2               -0.078649
Squat3               -0.076897
Squat1               -0.071948
Actual_Deadlift2     -0.060616
Actual_Squat3        -0.058863
Weight               -0.052530
Actual_Deadlift1     -0.052476
Name: Failed_Benchpress3, dtype: float64

In [87]:
x=data[['Age','Benchpress1','Benchpress2','Benchpress3','Failed_Benchpress1','Failed_Benchpress2']]
y=data['Failed_Benchpress3']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

In [ ]:
def objective(trail,data=x,target=y):
    param={
        "criterion":trail.suggest_categorical("criterion",["friedman_mse", "squared_error"]),
        "learning_rate":trail.suggest_categorical("learning_rate",[.00001,.0003,.008,.02,.01,1]),
        "n_estimators":trail.suggest_categorical('n_estimators',[50,100,300,500]),
        "max_depth":trail.suggest_categorical('max_depth',[8,16,32,64,128]),
        "min_samples_leaf":trail.suggest_categorical('min_samples_leaf',[8,16,32,64,128])
    }
    GB_status_bench3=GradientBoostingClassifier(**param)
    GB_status_bench3.fit(x_train,y_train)
    y_pred=GB_status_bench3.predict(x_test)
    return accuracy_score(y_test,y_pred)

trials=optuna.create_study(direction='maximize')
trials.optimize(objective,n_trials=5)

[I 2024-09-29 16:38:11,438] A new study created in memory with name: no-name-2084251e-d2e2-4e7f-ae32-b123c8f212b0
[I 2024-09-29 16:38:12,373] Trial 0 finished with value: 0.6096256684491979 and parameters: {'criterion': 'squared_error', 'learning_rate': 0.02, 'n_estimators': 500, 'max_depth': 16, 'min_samples_leaf': 128}. Best is trial 0 with value: 0.6096256684491979.
[I 2024-09-29 16:38:14,323] Trial 1 finished with value: 0.5882352941176471 and parameters: {'criterion': 'squared_error', 'learning_rate': 0.01, 'n_estimators': 300, 'max_depth': 16, 'min_samples_leaf': 8}. Best is trial 0 with value: 0.6096256684491979.
[I 2024-09-29 16:38:15,063] Trial 2 finished with value: 0.6363636363636364 and parameters: {'criterion': 'friedman_mse', 'learning_rate': 1e-05, 'n_estimators': 500, 'max_depth': 64, 'min_samples_leaf': 128}. Best is trial 2 with value: 0.6363636363636364.
[I 2024-09-29 16:38:15,287] Trial 3 finished with value: 0.5561497326203209 and parameters: {'criterion': 'friedma

In [89]:
params=trials.best_trial.params
GB_status_bench3=GradientBoostingClassifier(**params)
GB_status_bench3.fit(x_train,y_train)
print(f"Training score for Gradientboosting classifier: {GB_status_bench3.score(x_train,y_train)}")
y_pred=GB_status_bench3.predict(x_test)
print(f"Testing score for Gradientboosting classifier: {accuracy_score(y_test,y_pred)}")

Training score for Gradientboosting classifier: 0.5785234899328859
Testing score for Gradientboosting classifier: 0.6363636363636364


In [90]:
print(f"Classification report:{classification_report(y_test,y_pred,zero_division=1.0)}")

Classification report:              precision    recall  f1-score   support

           0       0.64      1.00      0.78       119
           1       1.00      0.00      0.00        68

    accuracy                           0.64       187
   macro avg       0.82      0.50      0.39       187
weighted avg       0.77      0.64      0.49       187



In [91]:
pickle.dump(GB_status_squat3,open('../Models/bench3_status_predictor.sav','wb'))

## Predicting if third lift is a successful lift (Deadlift)

In [53]:
correlation['Failed_Deadlift3'].sort_values(ascending=False)[:15]

Failed_Deadlift3      1.000000
Deadlift2             0.196466
Deadlift3             0.195023
Deadlift1             0.192379
Squat1                0.159495
Squat2                0.154503
Benchpress1           0.147590
Squat3                0.145374
Benchpress2           0.143579
Actual_Deadlift1      0.129020
Benchpress3           0.128056
Gender                0.119123
Total                 0.117910
Failed_Squat3         0.089634
Failed_Benchpress1    0.084418
Name: Failed_Deadlift3, dtype: float64

In [13]:
correlation['Failed_Deadlift3'].sort_values(ascending=False)[::-1][:10]

Actual_Deadlift3     -0.960759
Age                  -0.098046
Actual_Squat3        -0.078711
Failed_Deadlift1     -0.034987
Actual_Squat2        -0.016450
Actual_Benchpress3   -0.014503
Actual_Squat1        -0.003897
Failed_Benchpress2    0.008029
Actual_Deadlift2      0.010930
Actual_Benchpress1    0.023626
Name: Failed_Deadlift3, dtype: float64

In [77]:
x=data[['Failed_Benchpress2','Failed_Squat1','Failed_Squat2']]
y=data['Failed_Deadlift3']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

In [ ]:
def objective(trail,data=x,target=y):
    param={
        "criterion":trail.suggest_categorical("criterion",["friedman_mse", "squared_error"]),
        "learning_rate":trail.suggest_categorical("learning_rate",[.00001,.0003,.008,.02,.01,1]),
        "n_estimators":trail.suggest_categorical('n_estimators',[50,100,300,500]),
        "max_depth":trail.suggest_categorical('max_depth',[8,16,32,64,128]),
        "min_samples_leaf":trail.suggest_categorical('min_samples_leaf',[8,16,32,64,128])
    }
    GB_status_deadlift3=GradientBoostingClassifier(**param)
    GB_status_deadlift3.fit(x_train,y_train)
    y_pred=GB_status_deadlift3.predict(x_test)
    return accuracy_score(y_test,y_pred)

trials=optuna.create_study(direction='maximize')
trials.optimize(objective,n_trials=5)

[I 2024-10-26 21:44:48,901] A new study created in memory with name: no-name-1cf4ae6b-e84d-4eba-9ed8-f8c2934a5d59
[I 2024-10-26 21:44:48,956] Trial 0 finished with value: 0.5882352941176471 and parameters: {'criterion': 'squared_error', 'learning_rate': 1e-05, 'n_estimators': 50, 'max_depth': 8, 'min_samples_leaf': 32}. Best is trial 0 with value: 0.5882352941176471.
[I 2024-10-26 21:44:49,197] Trial 1 finished with value: 0.5882352941176471 and parameters: {'criterion': 'friedman_mse', 'learning_rate': 0.008, 'n_estimators': 300, 'max_depth': 64, 'min_samples_leaf': 8}. Best is trial 0 with value: 0.5882352941176471.
[I 2024-10-26 21:44:49,268] Trial 2 finished with value: 0.5882352941176471 and parameters: {'criterion': 'squared_error', 'learning_rate': 1e-05, 'n_estimators': 100, 'max_depth': 8, 'min_samples_leaf': 32}. Best is trial 0 with value: 0.5882352941176471.
[I 2024-10-26 21:44:49,629] Trial 3 finished with value: 0.5561497326203209 and parameters: {'criterion': 'friedman_m

In [74]:
params=trials.best_trial.params
GB_status_deadlift3=GradientBoostingClassifier(**params)
GB_status_deadlift3.fit(x_train,y_train)
print(f"Training score for Gradientboosting classifier: {GB_status_deadlift3.score(x_train,y_train)}")
y_pred=GB_status_deadlift3.predict(x_test)
print(f"Testing score for Gradientboosting classifier: {accuracy_score(y_test,y_pred)}")

Training score for Gradientboosting classifier: 0.5973154362416108
Testing score for Gradientboosting classifier: 0.5347593582887701


In [98]:
print(f"Classification report:{classification_report(y_test,y_pred,zero_division=1.0)}")

Classification report:              precision    recall  f1-score   support

           0       0.63      1.00      0.77       117
           1       1.00      0.00      0.00        70

    accuracy                           0.63       187
   macro avg       0.81      0.50      0.38       187
weighted avg       0.77      0.63      0.48       187



In [99]:
pickle.dump(GB_status_deadlift3,open('../Models/deadlift3_status_predictor.sav','wb'))